<a href="https://colab.research.google.com/github/Rawfulroufkhan/Rawfulroufkhan/blob/main/DNN_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [7]:
transforms_data = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}
dataset_root = 'plant_data'
if not os.path.exists(dataset_root):
    print(f"Creating dummy dataset directory: {dataset_root}")
    os.makedirs(os.path.join(dataset_root, 'train', 'healthy'), exist_ok=True)
    os.makedirs(os.path.join(dataset_root, 'train', 'diseased'), exist_ok=True)
    os.makedirs(os.path.join(dataset_root, 'val', 'healthy'), exist_ok=True)
    os.makedirs(os.path.join(dataset_root, 'val', 'diseased'), exist_ok=True)

    # Create dummy images (e.g., blank images)
    from PIL import Image
    dummy_img = Image.new('RGB', (224, 224), color = 'red')
    dummy_img.save(os.path.join(dataset_root, 'train', 'healthy', 'dummy_h1.png'))
    dummy_img = Image.new('RGB', (224, 224), color = 'green')
    dummy_img.save(os.path.join(dataset_root, 'train', 'diseased', 'dummy_d1.png'))
    dummy_img.save(os.path.join(dataset_root, 'val', 'healthy', 'dummy_h2.png'))
    dummy_img.save(os.path.join(dataset_root, 'val', 'diseased', 'dummy_d2.png'))
    print("Dummy dataset created. Replace with actual PlantVillage data for real training.")


image_datasets = {
    x: datasets.ImageFolder(os.path.join(dataset_root, x), transforms_data[x])
    for x in ['train', 'val']
}

dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2)
    for x in ['train', 'val']
}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Dataset sizes: {dataset_sizes}")
print(f"Class names: {class_names}")


Creating dummy dataset directory: plant_data
Dummy dataset created. Replace with actual PlantVillage data for real training.
Dataset sizes: {'train': 2, 'val': 2}
Class names: ['diseased', 'healthy']


In [ ]:
model_ft = models.resnet18(pretrained=True)
num_ftrs = model_ft.fc.in_features

model_ft.fc = nn.Linear(num_ftrs, len(class_names))

model_ft = model_ft.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)

In [4]:
def train_model(model, criterion, optimizer, num_epochs=2):
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward
                # track history only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    print('Training complete')
    return model

# Start training (using a small number of epochs for demonstration)
model_ft = train_model(model_ft, criterion, optimizer_ft, num_epochs=2)

NameError: name 'model_ft' is not defined

In [ ]:
def evaluate_model(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy of the model on the test images: {accuracy:.2f}%')


evaluate_model(model_ft, dataloaders['val'])